# 01.5 · 幂等设计（Idempotency）

> 消息重试场景下避免重复处理的必备模式。


In [ ]:
import nest_asyncio; nest_asyncio.apply()
import queue, threading, asyncio, time, uuid, random, copy
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any

# ── 模拟 LLM / 向量检索 ─────────────────────────────────────────────────────
CORPUS = [
    "RAG 通过检索外部文档来扩充 LLM 的上下文。",
    "向量数据库支持语义相似度搜索。",
    "Reranker 用交叉编码器对候选文档重新打分。",
    "幂等设计让系统在重试时不产生副作用。",
    "Backpressure 防止下游服务因流量过大而崩溃。",
    "可观测性包括 Metrics、Traces 和 Logs 三大支柱。",
    "Graceful degradation 让系统在部分故障时仍能返回有用结果。",
    "Hub-and-Spoke 模式由 Orchestrator 统一分派任务。",
    "Blackboard 模式通过共享工作区实现多 Agent 协作。",
    "消息队列解耦生产者和消费者，支持异步处理。",
]

def fake_retrieve(query: str, top_k: int = 5, latency: float = 0.3) -> list[dict]:
    """模拟向量检索，返回 top_k 相关文档"""
    time.sleep(latency)
    scored = [(doc, random.uniform(0.5, 1.0)) for doc in CORPUS]
    scored.sort(key=lambda x: -x[1])
    return [{"text": doc, "score": round(s, 3)} for doc, s in scored[:top_k]]

def fake_rerank(query: str, hits: list[dict], top_k: int = 3, latency: float = 0.2) -> list[dict]:
    """模拟 cross-encoder reranker"""
    time.sleep(latency)
    reranked = copy.deepcopy(hits)
    for h in reranked:
        h["score"] = round(h["score"] * random.uniform(0.9, 1.1), 3)
    reranked.sort(key=lambda x: -x["score"])
    return reranked[:top_k]

def fake_generate(question: str, context: list[dict], latency: float = 0.5) -> str:
    """模拟 LLM 生成"""
    time.sleep(latency)
    snippets = " ".join(h["text"][:30] for h in context[:2])
    return f"[答案] 关于「{question[:20]}」：{snippets}..."

def fake_verify(answer: str, context: list[dict], latency: float = 0.15) -> dict:
    """模拟答案校验"""
    time.sleep(latency)
    return {"grounded": random.random() > 0.2, "confidence": round(random.uniform(0.7, 1.0), 2)}

print("✅ 模拟工具加载完毕")

---
## Part 5 · 幂等设计（Idempotency）

**问题**：消息队列在网络故障或消费者崩溃后会**重复投递**，如果处理不是幂等的，就会产生重复写入或重复 API 调用。

**解决方案**：每条消息带 `request_id`，处理前检查是否已处理过。

In [ ]:
# ── 模拟幂等处理器 ───────────────────────────────────────────────────────────
class IdempotentProcessor:
    def __init__(self):
        self._processed: set[str] = set()    # 生产中用 Redis SET + TTL
        self._call_count = 0

    def process(self, job: dict) -> dict | None:
        rid = job["request_id"]

        # 幂等检查
        if rid in self._processed:
            print(f"  ⚠️  [{rid[:8]}] 重复消息，跳过（已处理过）")
            return None

        self._call_count += 1
        self._processed.add(rid)
        result = fake_generate(job["query"], fake_retrieve(job["query"], latency=0), latency=0)
        print(f"  ✅ [{rid[:8]}] 处理完成（第 {self._call_count} 次实际处理）")
        return {"result": result}


processor = IdempotentProcessor()

# ── 模拟场景：同一条消息被投递 3 次（网络重试）───────────────────────────────
job = {"request_id": str(uuid.uuid4()), "query": "幂等性是什么？"}

print("=== 模拟消息重复投递 ===")
for i in range(3):
    print(f"\n投递第 {i+1} 次：")
    processor.process(job)

print(f"\n实际处理次数：{processor._call_count}（应为 1，不是 3）")

# ── 不同 request_id 的新消息正常处理 ─────────────────────────────────────────
print("\n=== 新消息（不同 request_id）===")
for q in ["什么是 RAG？", "什么是向量数据库？"]:
    new_job = {"request_id": str(uuid.uuid4()), "query": q}
    processor.process(new_job)

print(f"\n总实际处理次数：{processor._call_count}（3 条唯一消息）")

## 📖 讲义

# Part 5
## 生产级要点

---

# 可观测性：三大支柱

<div class="columns">
<div>

### Metrics（指标）
```python
from prometheus_client import Counter, Histogram

requests_total = Counter(
    "agent_requests_total",
    "Total requests",
    ["agent_name", "status"]
)
latency = Histogram(
    "agent_latency_seconds",
    "Request latency",
    ["agent_name"]
)

# 使用
with latency.labels("retriever").time():
    hits = do_retrieval(query)
requests_total.labels("retriever", "success").inc()
```

</div>
<div>

### Traces（链路追踪）
```python
from opentelemetry import trace

tracer = trace.get_tracer("retriever")

with tracer.start_as_current_span("retrieve") as span:
    span.set_attribute("query", query)
    span.set_attribute("top_k", 20)
    hits = do_retrieval(query)
    span.set_attribute("hits_count", len(hits))
```

### Logs（结构化日志）
```python
import structlog
log = structlog.get_logger()

log.info("retrieval_done",
    request_id=job["request_id"],
    query=query,
    hits=len(hits),
    latency_ms=elapsed)
```

</div>
</div>

---

# 安全隔离与最小权限

**Agent 安全清单**

| 层次 | 措施 |
|------|------|
| **网络** | 各 Agent 只开放必要端口，使用 mTLS |
| **IAM** | 每个 Agent 独立 Service Account，最小权限 |
| **输入校验** | 所有入参用 Pydantic / jsonschema 校验 |
| **输出过滤** | LLM 输出过滤 Prompt Injection / PII |
| **速率限制** | 每个 Agent 入口限流（令牌桶） |
| **审计日志** | 记录所有 tool call 与外部 API 调用 |

> **关键原则**：绝不直接执行 LLM 生成的命令，必须经过参数白名单校验

---

# 成本优化策略

<div class="columns">
<div>

### 分层模型路由
```python
def select_model(task_type: str) -> str:
    routing = {
        "retrieval_score":  "haiku-4-5",   # 便宜
        "rerank":           "haiku-4-5",
        "generation_simple":"sonnet-4-6",  # 主力
        "generation_complex":"opus-4-6",   # 贵，按需
        "verification":     "haiku-4-5",
    }
    return routing.get(task_type, "sonnet-4-6")
```

</div>
<div>

### 缓存策略
```python
import hashlib

def cached_retrieve(query: str):
    key = f"cache:retrieve:{hashlib.md5(query.encode()).hexdigest()}"
    cached = redis.get(key)
    if cached:
        return json.loads(cached)
    result = do_retrieval(query)
    redis.setex(key, 3600, json.dumps(result))
    return result
```

### 批处理
- 聚合多个请求一次调用 LLM
- 使用 Embedding batch API

</div>
</div>

---

# Graceful Degradation（优雅降级）

```python
# orchestrator_with_fallback.py
async def handle_query_with_fallback(query: str):
    try:
        # 正常路径：完整 Pipeline
        hits  = await retriever.retrieve(query)
        topk  = await reranker.rerank(query, hits)   # 可能失败
        answer = await generator.generate(query, topk)
    except RerankerUnavailable:
        # 降级：跳过 Reranker，直接用粗检结果
        logger.warning("reranker_down_fallback", query=query)
        answer = await generator.generate(query, hits[:5])
    except GeneratorTimeout:
        # 再降级：直接返回检索片段
        return {"answer": None, "snippets": hits[:3], "degraded": True}

    return {"answer": answer, "degraded": False}
```

**降级层次**：完整 Pipeline → 跳过 Reranker → 仅返回检索片段 → 缓存结果 → 静态兜底

---

---
**导航**：[← 01.4 · Blackboard 模式](./01.4_blackboard_mode.ipynb) · [📋 课程索引](./01_multi_agent_arch_demo.ipynb) · [01.6 · 可观测性 →](./01.6_observability.ipynb)
